In [1]:
#第17章/加载数据集
from datasets import load_dataset
import torchvision
import torch


#加载全部数据到内存中
def get_data(cls):
    #加载数据集
    dataset = load_dataset('lansinuote/gen.5.flower.book', split='train')

    #根据字段过滤
    def f(data):
        return [i == cls for i in data['cls']]

    dataset = dataset.filter(f, batched=True, batch_size=100, num_proc=1)

    #删除多余的字段
    dataset = dataset.remove_columns(['cls'])

    #图像数据预处理
    compose = torchvision.transforms.Compose([
        torchvision.transforms.Resize(128),
        torchvision.transforms.ToTensor(),
        lambda x: x * 2 - 1,
    ])

    def f(data):
        image = compose(data['image'][0]).unsqueeze(dim=0)
        return {'image': image}

    dataset = dataset.with_transform(f)

    #为了加速数据遍历的效率,把全体数据载入内存以加速IO
    dataset_tensor = torch.empty(len(dataset), 3, 128, 128)

    for i in range(len(dataset)):
        dataset_tensor[i] = dataset[i]['image']

    return dataset_tensor


data_A = get_data(cls=0)
data_B = get_data(cls=1)

data_A.shape, data_B.shape, data_A.dtype

Using custom data configuration lansinuote--gen.5.flower.book-34a531175c385260
Found cached dataset parquet (/root/.cache/huggingface/datasets/lansinuote___parquet/lansinuote--gen.5.flower.book-34a531175c385260/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec)


  0%|          | 0/20 [00:00<?, ?ba/s]

Using custom data configuration lansinuote--gen.5.flower.book-34a531175c385260
Found cached dataset parquet (/root/.cache/huggingface/datasets/lansinuote___parquet/lansinuote--gen.5.flower.book-34a531175c385260/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec)


  0%|          | 0/20 [00:00<?, ?ba/s]

(torch.Size([1000, 3, 128, 128]),
 torch.Size([1000, 3, 128, 128]),
 torch.float32)

In [2]:
#第17章/定义loader
import torch

loader_A = torch.utils.data.DataLoader(dataset=data_A,
                                       batch_size=1,
                                       shuffle=True)

loader_B = torch.utils.data.DataLoader(dataset=data_B,
                                       batch_size=1,
                                       shuffle=True)

len(loader_A), len(loader_B), next(iter(loader_A)).shape

(1000, 1000, torch.Size([1, 3, 128, 128]))

In [3]:
#第17章/show函数
def show(images):
    from matplotlib import pyplot as plt

    images = images.to('cpu').detach()[:50]
    images = images.permute(0, 2, 3, 1)
    images = (images + 1) / 2

    plt.figure(figsize=(20, 10))

    for i in range(len(images)):
        plt.subplot(5, 10, i + 1)
        plt.imshow(images[i])
        plt.axis('off')

    plt.show()


def stack(loader, n):
    datas = []
    for _ in range(n):
        datas.append(next(iter(loader)))
    return torch.cat(datas, dim=0)


show(stack(loader_A, 20))
show(stack(loader_B, 20))

<Figure size 2000x1000 with 20 Axes>

<Figure size 2000x1000 with 20 Axes>

In [4]:
#第17章/定义CLS模型
def get_cls():
    return torch.nn.Sequential(
        torch.nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
        torch.nn.LeakyReLU(0.2),
        torch.nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
        torch.nn.InstanceNorm2d(num_features=64),
        torch.nn.LeakyReLU(0.2),
        torch.nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
        torch.nn.InstanceNorm2d(num_features=128),
        torch.nn.LeakyReLU(0.2),
        torch.nn.Conv2d(128, 256, kernel_size=4, stride=1, padding=1),
        torch.nn.InstanceNorm2d(num_features=256),
        torch.nn.LeakyReLU(0.2),
        torch.nn.Conv2d(256, 1, kernel_size=4, stride=1, padding=1),
    )


cls_A = get_cls()
cls_B = get_cls()

cls_A(torch.randn(2, 3, 128, 128)).shape

torch.Size([2, 1, 14, 14])

In [5]:
#第17章/定义GEN模型
class UNet(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.down = torch.nn.ModuleList([
            torch.nn.Sequential(
                torch.nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
                torch.nn.InstanceNorm2d(num_features=32),
                torch.nn.ReLU(),
            ),
            torch.nn.Sequential(
                torch.nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
                torch.nn.InstanceNorm2d(num_features=64),
                torch.nn.ReLU(),
            ),
            torch.nn.Sequential(
                torch.nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
                torch.nn.InstanceNorm2d(num_features=128),
                torch.nn.ReLU(),
            ),
            torch.nn.Sequential(
                torch.nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
                torch.nn.InstanceNorm2d(num_features=256),
                torch.nn.ReLU(),
            ),
        ])

        self.up = torch.nn.ModuleList([
            torch.nn.Sequential(
                torch.nn.UpsamplingNearest2d(size=16),
                torch.nn.Conv2d(256, 128, kernel_size=3, stride=1, padding=1),
                torch.nn.InstanceNorm2d(num_features=128),
                torch.nn.ReLU(),
            ),
            torch.nn.Sequential(
                torch.nn.UpsamplingNearest2d(size=32),
                torch.nn.Conv2d(256, 64, kernel_size=3, stride=1, padding=1),
                torch.nn.InstanceNorm2d(num_features=64),
                torch.nn.ReLU(),
            ),
            torch.nn.Sequential(
                torch.nn.UpsamplingNearest2d(size=64),
                torch.nn.Conv2d(128, 32, kernel_size=3, stride=1, padding=1),
                torch.nn.InstanceNorm2d(num_features=32),
                torch.nn.ReLU(),
            ),
            torch.nn.Sequential(
                torch.nn.UpsamplingNearest2d(size=128),
                torch.nn.Conv2d(64, 3, kernel_size=3, stride=1, padding=1),
                torch.nn.Tanh(),
            ),
        ])

    def forward(self, x):
        down_out = []
        for layer in self.down:
            x = layer(x)
            down_out.append(x)

        up_out = torch.cat((self.up[0](down_out[3]), down_out[2]), dim=1)
        up_out = torch.cat((self.up[1](up_out), down_out[1]), dim=1)
        up_out = torch.cat((self.up[2](up_out), down_out[0]), dim=1)
        up_out = self.up[3](up_out)

        return up_out


gen_A = UNet()
gen_B = UNet()

gen_A(torch.randn(2, 3, 128, 128)).shape

torch.Size([2, 3, 128, 128])

In [6]:
#第17章/初始化工具类
def set_requires_grad(model, requires_grad):
    for param in model.parameters():
        param.requires_grad_(requires_grad)


optimizer_cls_A = torch.optim.Adam(cls_A.parameters(), lr=2e-4)
optimizer_cls_B = torch.optim.Adam(cls_B.parameters(), lr=2e-4)

optimizer_gen_A = torch.optim.Adam(gen_A.parameters(), lr=2e-4)
optimizer_gen_B = torch.optim.Adam(gen_B.parameters(), lr=2e-4)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

cls_A.to(device).train()
cls_B.to(device).train()
gen_A.to(device).train()
gen_B.to(device).train()

criterion_mse = torch.nn.MSELoss()
criterion_l1 = torch.nn.L1Loss()

device

'cuda'

In [7]:
#第17章/训练CLS模型的函数
def train_cls():
    set_requires_grad(cls_A, True)
    set_requires_grad(cls_B, True)
    set_requires_grad(gen_A, False)
    set_requires_grad(gen_B, False)

    def update(cls, optimizer, image, label):
        pred = cls(image)
        label = torch.full((1, 1, 14, 14), label, device=device).float()
        loss = criterion_mse(pred, label)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        return loss.item()

    image_A = next(iter(loader_A)).to(device)
    image_B = next(iter(loader_B)).to(device)

    with torch.no_grad():
        image_A_gen = gen_A(image_B)
        image_B_gen = gen_B(image_A)

    loss_sum = 0

    loss = update(cls_A, optimizer_cls_A, image_A, 1)
    loss_sum += loss

    loss = update(cls_A, optimizer_cls_A, image_A_gen, 0)
    loss_sum += loss

    loss = update(cls_B, optimizer_cls_B, image_B, 1)
    loss_sum += loss

    loss = update(cls_B, optimizer_cls_B, image_B_gen, 0)
    loss_sum += loss

    return loss_sum / 4


train_cls()

0.659519612789154

In [8]:
#第17章/训练GEN模型的函数
def train_gen():
    set_requires_grad(cls_A, False)
    set_requires_grad(cls_B, False)
    set_requires_grad(gen_A, True)
    set_requires_grad(gen_B, True)

    image_A = next(iter(loader_A)).to(device)
    image_B = next(iter(loader_B)).to(device)

    #A变B,B变A
    image_gen_B = gen_B(image_A)
    image_gen_A = gen_A(image_B)

    losses = []

    #计算转换的成绩
    loss = criterion_mse(cls_A(image_gen_A),
                         torch.ones(1, 1, 14, 14, device=device))
    losses.append(loss)

    loss = criterion_mse(cls_B(image_gen_B),
                         torch.ones(1, 1, 14, 14, device=device))
    losses.append(loss)

    #把转换过的图片再转换回来
    loss = criterion_l1(gen_A(image_gen_B), image_A) * 10
    losses.append(loss)

    loss = criterion_l1(gen_B(image_gen_A), image_B) * 10
    losses.append(loss)

    #A变A,B变B,这次转换应该是不变的
    loss = criterion_l1(gen_A(image_A), image_A) * 2
    losses.append(loss)

    loss = criterion_l1(gen_B(image_B), image_B) * 2
    losses.append(loss)

    loss = sum(losses)
    loss.backward()

    optimizer_gen_A.step()
    optimizer_gen_B.step()

    optimizer_gen_A.zero_grad()
    optimizer_gen_B.zero_grad()

    return loss.item()


train_gen()

15.228337287902832

In [9]:
#第17章/训练
def train():
    for epoch in range(4_0000):
        loss_cls = train_cls()
        loss_gen = train_gen()

        if epoch % 8000 == 0:
            print(epoch, loss_cls, loss_gen)

            image_A = stack(loader_A, 10)
            image_B = stack(loader_B, 10)

            print('A')
            show(image_A)

            print('A to B')
            show(gen_B(image_A.to(device)))

            print('B')
            show(image_B)

            print('B to A')
            show(gen_A(image_B.to(device)))

    torch.save(cls_A.to('cpu'), 'save/cls_A.model')
    torch.save(cls_B.to('cpu'), 'save/cls_B.model')
    torch.save(gen_A.to('cpu'), 'save/gen_A.model')
    torch.save(gen_B.to('cpu'), 'save/gen_B.model')


train()

0 0.4870578348636627 13.631972312927246
A


<Figure size 2000x1000 with 10 Axes>

A to B


<Figure size 2000x1000 with 10 Axes>

B


...

<Figure size 2000x1000 with 10 Axes>

B


<Figure size 2000x1000 with 10 Axes>

B to A


<Figure size 2000x1000 with 10 Axes>

In [10]:
#第17章/测试
def test():
    with torch.no_grad():
        image_A = stack(loader_A, 10)
        print('A')
        show(image_A)

        print('A to B')
        show(gen_B(image_A))

        image_B = stack(loader_B, 10)
        print('B')
        show(image_B)

        print('B to A')
        show(gen_A(image_B))


gen_A = torch.load('save/gen_A.model')
gen_B = torch.load('save/gen_B.model')

test()

A


<Figure size 2000x1000 with 10 Axes>

A to B


<Figure size 2000x1000 with 10 Axes>

B


<Figure size 2000x1000 with 10 Axes>

B to A


<Figure size 2000x1000 with 10 Axes>

In [11]:
#第17章/在线加载笔者训练好的模型并测试
from transformers import PreTrainedModel, PretrainedConfig


class Model(PreTrainedModel):
    config_class = PretrainedConfig

    def __init__(self, config):
        super().__init__(config)
        self.cls_A = cls_A.to('cpu')
        self.cls_B = cls_B.to('cpu')
        self.gen_A = gen_A.to('cpu')
        self.gen_B = gen_B.to('cpu')


#加载训练好的模型
model = Model.from_pretrained('lansinuote/gen.6.cyclegan.book')
gen_A = model.gen_A
gen_B = model.gen_B

test()

A


<Figure size 2000x1000 with 10 Axes>

A to B


<Figure size 2000x1000 with 10 Axes>

B


<Figure size 2000x1000 with 10 Axes>

B to A


<Figure size 2000x1000 with 10 Axes>